### Notebook 04: Distributed Training with `pyspark.ml.connect`

In this notebook, we revisit the goal of training a model on Databricks compute using Spark Connect — this time using the newer `pyspark.ml.connect` API. This API was built specifically for Spark Connect and works without needing a JVM or `SparkContext` on the client.

We’ll train a logistic regression model using distributed execution on your Databricks cluster, log everything to MLflow, and register the result to Unity Catalog. This is just like in earlier notebooks, but fully remote.

Let’s see how far the new Connect-native API can take us!

👉 [Learn more about distributed ML with Spark Connect](https://docs.databricks.com/aws/en/machine-learning/train-model/distributed-training/distributed-ml-for-spark-connect)



In [1]:
# Set up Databricks Connect session
from databricks.connect import DatabricksSession
import os
import config

# Pull connection values from config file
os.environ["DATABRICKS_HOST"] = config.DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = config.DATABRICKS_TOKEN
os.environ["DATABRICKS_CLUSTER_ID"] = config.DATABRICKS_CLUSTER_ID
os.environ["DATABRICKS_WORKSPACE_ID"] = config.DATABRICKS_WORKSPACE_ID

# Initialize Spark session routed through Databricks Connect
spark = DatabricksSession.builder.getOrCreate()


# ML Example

In [2]:
# Import PySpark ML components and supporting libraries
from pyspark.ml.connect.classification import LogisticRegression
from pyspark.ml.connect.feature import StandardScaler
from pyspark.ml.connect.pipeline import Pipeline
from pyspark.ml.connect.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.connect.tuning import CrossValidator
from pyspark.ml.tuning import ParamGridBuilder

# MLflow and utility imports
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.exceptions import MlflowException
from pyspark.sql.functions import array, col
from datetime import datetime

In [3]:
# Define catalog, schema, and table names for reading data and registering the model
CATALOG_NAME = "alexander_booth"
SCHEMA_NAME = "default"
TABLE_NAME = "breast_cancer_training_data"
MODEL_NAME = "breast_cancer_model"

In [4]:
# Build resource URIs for the table and model
input_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
target_col = "label"
uc_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"

# Use current date for experiment naming
today = datetime.now().strftime("%Y%m%d")
experiment_name = f"breast_cancer_{today}"

In [5]:
# Configure MLflow to use Unity Catalog as registry backend
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Define path and artifact location for experiment
experiment_path = f"/Users/alexander.booth@databricks.com/{experiment_name}"

# Log artifacts to a Volume
artifact_location = "dbfs:/Volumes/alexander_booth/default/mlflow_artifacts" 

# Create experiment if it doesn't exist
try:
    mlflow.create_experiment(name=experiment_path, artifact_location=artifact_location)
except MlflowException:
    pass

mlflow.set_experiment(experiment_path)

<Experiment: artifact_location='dbfs:/Volumes/alexander_booth/default/mlflow_artifacts', creation_time=1747420714817, experiment_id='913238801103218', last_update_time=1747420722183, lifecycle_stage='active', name='/Users/alexander.booth@databricks.com/breast_cancer_20250516', tags={'mlflow.experiment.sourceName': '/Users/alexander.booth@databricks.com/breast_cancer_20250516',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'alexander.booth@databricks.com',
 'mlflow.ownerId': '8950113264559567'}>

In [6]:
# Read raw training data from Unity Catalog
df_raw = spark.read.table(input_table).dropna()

# Convert features into a single array column (required by MLlib)
label_col = "label"
feature_cols = [c for c in df_raw.columns if c != label_col]

# Build vectorized dataframe
df_vectorized = df_raw.select(
    col(label_col).cast("double").alias("label"),
    array(*[col(c) for c in feature_cols]).alias("features")
)

# Split data into training and test sets
train_df, test_df = df_vectorized.randomSplit([0.8, 0.2], seed=42)
train_df.show(5)

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|[10.95, 21.35, 71...|
|  0.0|[11.08, 18.83, 73...|
|  0.0|[11.76, 18.14, 75...|
|  0.0|[11.8, 16.58, 78....|
|  0.0|[11.84, 18.7, 77....|
+-----+--------------------+
only showing top 5 rows


In [7]:
import logging

# Silence the log streaming server warnings
logger = logging.getLogger("pyspark.ml.torch")
logger.setLevel(logging.FATAL)
logger.propagate = False  # prevent bubbling up to root logger

In [8]:
# --- Train Logistic Regression using pyspark.ml.connect ---

# Define pipeline stages
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
lr = LogisticRegression(numTrainWorkers=2, featuresCol="scaled_features")
pipeline = Pipeline(stages=[scaler, lr])

# Define grid search parameters
grid = ParamGridBuilder().addGrid(lr.maxIter, [2, 200]).build()

# Wrap pipeline in cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=grid,
    parallelism=2,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
)

# Start MLflow run
now = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"logistic-regression-{now}"

with mlflow.start_run(run_name=run_name) as run:
    # Optional: mlflow.pyspark.ml.connect.autolog() once stable
    model = cv.fit(train_df)
    predictions = model.transform(test_df)

    # Log evaluation metrics
    binary_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    mlflow.log_metric("areaUnderROC", binary_eval.evaluate(predictions))

    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    mlflow.log_metric("accuracy", evaluator.evaluate(predictions))

    # Log best model parameters
    best_model = model.bestModel.stages[1]  # stage[0] = scaler, stage[1] = logistic regression
    for param, value in best_model.extractParamMap().items():
        mlflow.log_param(param.name, value)

    # Infer schema for input/output logging
    sample_input = test_df.limit(5).toPandas()
    sample_output = predictions.limit(5).toPandas()[["prediction"]]
    signature = infer_signature(sample_input, sample_output)

    # Log model to Unity Catalog-compatible location
    mlflow.spark.log_model(model.bestModel, artifact_path="model", signature=signature)

    # Register model in Unity Catalog
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print(f"Run ID: {run_id}")
    print(f"Model URI: {model_uri}")

    mlflow.register_model(model_uri=model_uri, name=uc_model_name)

# Note: You may see harmless logger errors or warnings. These can be safely ignored.

--- Logging error ---
Traceback (most recent call last):
  File "/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/ml/torch/distributor.py", line 768, in _run_distributed_training
    log_streaming_server.start(spark_host_address=self.driver_address)
  File "/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/ml/torch/log_communication.py", line 84, in start
    self.port = LogStreamingServer._get_free_port(spark_host_address)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/ml/torch/log_communication.py", line 70, in _get_free_port
    tcp.bind((spark_host_address, 0))
OSError: [Errno 49] Can't assign requested address

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/logging/__init

Run ID: 5f785e7e8f534154b2c4cf730c4627e7
Model URI: runs:/5f785e7e8f534154b2c4cf730c4627e7/model


Registered model 'alexander_booth.default.breast_cancer_model' already exists. Creating a new version of this model...
Created version '4' of model 'alexander_booth.default.breast_cancer_model'.


🏃 View run logistic-regression-20250516_133926 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/913238801103218/runs/5f785e7e8f534154b2c4cf730c4627e7
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/913238801103218


---

### Wrapping Up: Distributed ML with Spark Connect

In this notebook, we successfully trained a logistic regression model using `pyspark.ml.connect`, running distributed training on a Databricks cluster via Spark Connect. We logged the entire experiment to MLflow and registered the model to Unity Catalog — all without needing a local JVM.

While `pyspark.ml.connect` currently supports a limited set of algorithms and transformers, it provides a viable path for distributed ML workflows in Spark Connect environments.

Looking ahead, Spark 4.0 aims to bring full support for the classic `pyspark.ml` API in Spark Connect, eliminating the need for separate APIs and simplifying the development experience.

For more information, see the Databricks documentation on [distributed ML with Spark Connect](https://docs.databricks.com/aws/en/machine-learning/train-model/distributed-training/distributed-ml-for-spark-connect) and the active development within the [Apache Spark](https://issues.apache.org/jira/browse/SPARK-49907) project.
